In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'
_os.environ['AWS_REGION'] = 'us-west-2'
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker Lineage Tracking - V3 SDK Example

This notebook demonstrates how to use SageMaker Lineage Tracking with the V3 Python SDK.

Amazon SageMaker Lineage enables events that happen within SageMaker to be traced via a graph structure. The data simplifies generating reports, making comparisons, or discovering relationships between events.

## What you will learn

- Create and manage lineage Contexts, Actions, and Artifacts
- Create Associations to link entities into a lineage graph
- Traverse associations to discover relationships
- Clean up lineage data

## Setup

Initialize a SageMaker session using the V3 `Session` class.

In [2]:
import boto3
from sagemaker.core.helper.session_helper import Session

region = boto3.Session().region_name
sagemaker_session = Session()
default_bucket = sagemaker_session.default_bucket()

print(f"Region: {region}")
print(f"Default bucket: {default_bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


Region: us-west-2
Default bucket: sagemaker-us-west-2-674622101542


In [3]:
from datetime import datetime
from sagemaker.core.lineage.context import Context
from sagemaker.core.lineage.action import Action
from sagemaker.core.lineage.association import Association
from sagemaker.core.lineage.artifact import Artifact

unique_id = str(int(datetime.now().replace(microsecond=0).timestamp()))
print(f"Unique id is {unique_id}")

Unique id is 1784619225


## Use Case 1: Create a Lineage Context

Contexts provide a method to logically group other lineage entities. The context name must be unique across all other contexts.

In [4]:
context_name = f"machine-learning-workflow-{unique_id}"

ml_workflow_context = Context.create(
    context_name=context_name,
    context_type="MLWorkflow",
    source_uri=unique_id,
    properties={"example": "true"},
)

print(f"Created context: {ml_workflow_context.context_name}")
print(f"Context ARN: {ml_workflow_context.context_arn}")

Created context: machine-learning-workflow-1784619225
Context ARN: arn:aws:sagemaker:us-west-2:674622101542:context/machine-learning-workflow-1784619225


## Use Case 2: List Contexts

Enumerate existing contexts sorted by creation time.

In [5]:
contexts = Context.list(sort_by="CreationTime", sort_order="Descending")

for ctx in contexts:
    print(ctx.context_name)

machine-learning-workflow-1784619225
data-catalog-demo-no-table-1784618212-86f5-1784618212-feature-group-pipeline-version
data-catalog-demo-no-table-1784618212-86f5-1784618212-feature-group-pipeline
data-catalog-demo-no-table-1784618212-86f5-1784618212
lakeformation-managed-fg-1784618205-feature-group-pipeline-version
lakeformation-managed-fg-1784618205
lakeformation-managed-fg-1784618205-feature-group-pipeline
data-catalog-demo-byot-glue-1784618191-681c-1784618191
data-catalog-demo-byot-glue-1784618191-681c-1784618191-feature-group-pipeline
data-catalog-demo-byot-glue-1784618191-681c-1784618191-feature-group-pipeline-version
data-catalog-demo-custom-iceberg-1784618169-e742-1784618169-feature-group-pipeline-version
data-catalog-demo-custom-iceberg-1784618169-e742-1784618169
data-catalog-demo-custom-iceberg-1784618169-e742-1784618169-feature-group-pipeline
data-catalog-demo-custom-glue-1784618147-2e8e-1784618147
data-catalog-demo-custom-glue-1784618147-2e8e-1784618147-feature-group-pipe

bankmrkt-HPO-4228d4-06Feb2024GzU-011-28d79205-DG-Task-EP-DG-1707251131-aws-endpoint
retail-recommendation-2024-02-06-19-59-1707250174-aws-model-package-group
pythia-12b-sft-2024-02-06-20-00-39-1707249641-aws-endpoint
ep-bankmrkt-hpo-dcbd97-02feb2024-datamodel-2-1706912899-aws-endpoint
ep-bankmrkt-hpo-dcbd97-02feb2024-inf-1-1706912898-aws-endpoint
ep-bankmrkt-hpo-dcbd97-02feb2024-datamodel-0-1706912896-aws-endpoint
ep-bankmrkt-ens-dcbd97-02feb2024-inf-0-1706912702-aws-endpoint
bankmrkt-HPO-dcbd97-02Feb20240SZ-001-20809c5b-DG-Task-EP-DG-1706911885-aws-endpoint
retail-recommendation-2024-02-02-21-42-1706910717-aws-model-package-group
ep-bankmrkt-hpo-016e4a-02feb2024-datamodel-2-1706909818-aws-endpoint
ep-bankmrkt-hpo-016e4a-02feb2024-inf-1-1706909817-aws-endpoint
ep-bankmrkt-hpo-016e4a-02feb2024-datamodel-0-1706909815-aws-endpoint
ep-bankmrkt-ens-016e4a-02feb2024-inf-0-1706909621-aws-endpoint
bankmrkt-HPO-016e4a-02Feb2024ltx-009-e5a8d109-DG-Task-EP-DG-1706908856-aws-endpoint
retail-recomm

ep-bankmrkt-hpo-1d54b8-25aug2023-datamodel-0-1692977754-aws-endpoint
ep-bankmrkt-ens-1d54b8-25aug2023-inf-0-1692977500-aws-endpoint
bankmrkt-HPO-1d54b8-25Aug2023nnn-004-f4338207-DG-Task-EP-DG-1692976758-aws-endpoint
jumpstart-example-FT-tensorflow-ic-imag-2023-08-25-15-16-19-131-1692976582-aws-endpoint
retail-recommendation-2023-08-25-14-49-1692975666-aws-model-package-group
jumpstart-example-tensorflow-ic-imagene-2023-08-25-14-59-45-445-1692975591-aws-endpoint
ep-bankmrkt-hpo-903cd4-25aug2023-datamodel-2-1692955555-aws-endpoint
ep-bankmrkt-hpo-903cd4-25aug2023-inf-1-1692955553-aws-endpoint
ep-bankmrkt-hpo-903cd4-25aug2023-datamodel-0-1692955552-aws-endpoint
ep-bankmrkt-ens-903cd4-25aug2023-inf-0-1692955248-aws-endpoint
bankmrkt-HPO-903cd4-25Aug2023WED-011-d2a33af8-DG-Task-EP-DG-1692954615-aws-endpoint
jumpstart-example-FT-tensorflow-ic-imag-2023-08-25-09-02-44-635-1692954168-aws-endpoint
jumpstart-example-tensorflow-ic-imagene-2023-08-25-08-46-39-940-1692953205-aws-endpoint
retail-rec

ep-bankmrkt-hpo-3e7751-07jul2023-inf-1-1688760037-aws-endpoint
ep-bankmrkt-hpo-3e7751-07jul2023-datamodel-0-1688760036-aws-endpoint
ep-bankmrkt-ens-3e7751-07jul2023-inf-0-1688759827-aws-endpoint
tensorflow-inference-ml-c4-2023-07-07-19-49-49-609-1688759390-aws-endpoint
jumpstart-example-FT-tensorflow-ic-imag-2023-07-07-19-44-32-113-1688759075-aws-endpoint
bankmrkt-HPO-3e7751-07Jul20239Pn-013-83b8d7a2-DG-Task-EP-DG-1688759013-aws-endpoint
tensorflow-inference-2023-07-07-19-38-22-189-1688758702-aws-endpoint
jumpstart-example-tensorflow-ic-imagene-2023-07-07-19-29-37-001-1688758182-aws-endpoint
retail-recommendation-2023-07-07-19-15-1688757908-aws-model-package-group
ep-bankmrkt-hpo-d3f47c-06jul2023-datamodel-2-1688676755-aws-endpoint
ep-bankmrkt-hpo-d3f47c-06jul2023-inf-1-1688676753-aws-endpoint
ep-bankmrkt-hpo-d3f47c-06jul2023-datamodel-0-1688676752-aws-endpoint
ep-bankmrkt-ens-d3f47c-06jul2023-inf-0-1688676508-aws-endpoint
ep-bankmrkt-hpo-958436-06jul2023-datamodel-2-1688676304-aws-end

retail-recommendation-2023-06-30-17-51-1688148077-aws-model-package-group
jumpstart-example-FT-tensorflow-ic-imag-2023-06-30-17-59-12-648-1688147956-aws-endpoint
tensorflow-inference-2023-06-30-17-57-03-684-1688147824-aws-endpoint
retail-recommendation-2023-06-30-17-38-1688147312-aws-model-package-group
jumpstart-example-tensorflow-ic-imagene-2023-06-30-17-44-38-497-1688147084-aws-endpoint
ep-bankmrkt-hpo-eb5ef4-28jun2023-datamodel-2-1687996342-aws-endpoint
ep-bankmrkt-hpo-eb5ef4-28jun2023-inf-1-1687996341-aws-endpoint
ep-bankmrkt-hpo-eb5ef4-28jun2023-datamodel-0-1687996340-aws-endpoint
ep-bankmrkt-ens-eb5ef4-28jun2023-inf-0-1687996147-aws-endpoint
ep-bankmrkt-hpo-2c94ed-28jun2023-datamodel-2-1687995914-aws-endpoint
ep-bankmrkt-hpo-2c94ed-28jun2023-inf-1-1687995912-aws-endpoint
ep-bankmrkt-hpo-2c94ed-28jun2023-datamodel-0-1687995911-aws-endpoint
ep-bankmrkt-ens-2c94ed-28jun2023-inf-0-1687995668-aws-endpoint
tensorflow-inference-ml-c4-2023-06-28-23-32-58-842-1687995179-aws-endpoint
jump

...

jumpstart-example-FT-tensorflow-ic-imag-2023-04-10-21-59-22-380-1681163965-aws-endpoint
tensorflow-inference-2023-04-10-21-54-51-840-1681163692-aws-endpoint
xgboost-2023-04-10-21-49-52-564-1681163393-aws-endpoint
xgboost-2023-04-10-21-47-17-394-1681163237-aws-endpoint
jumpstart-example-tensorflow-ic-imagene-2023-04-10-21-45-39-890-1681163145-aws-endpoint
retail-recommendation-2023-04-10-21-34-1681163034-aws-model-package-group
xgboost-2023-04-10-21-43-43-850-1681163024-aws-endpoint
triton-resnet-pt-2023-04-10-21-43-18-1681162998-aws-endpoint
ep-bankmrkt-hpo-983f6e-10apr2023-datamodel-2-1681160852-aws-endpoint
ep-bankmrkt-hpo-983f6e-10apr2023-inf-1-1681160851-aws-endpoint
ep-bankmrkt-hpo-983f6e-10apr2023-datamodel-0-1681160849-aws-endpoint
ep-bankmrkt-ens-983f6e-10apr2023-inf-0-1681160606-aws-endpoint
bankmrkt-HPO-983f6e-10Apr2023fad-012-5f1f0eeb-DG-Task-EP-DG-1681159692-aws-endpoint
tensorflow-inference-ml-c4-2023-04-10-20-43-17-425-1681159397-aws-endpoint
jumpstart-example-FT-tensorfl

linear-learner-2023-03-19-17-43-48-951-1679247830-aws-endpoint
jumpstart-example-FT-tensorflow-od1-ssd-2023-03-19-17-35-30-204-1679247351-aws-endpoint
forecasting-deepar-230319-1657-007-56a7917d-1679247275-aws-endpoint
linear-learner-2023-03-19-17-32-10-104-1679247131-aws-endpoint
sm-clarify-DEMO-clarify-model-19-03-2023-17-10--1679246646-fb61-1679246646-aws-endpoint
sm-clarify-DEMO-xgb-churn-pred-model-monitor-20-1679246479-6c4e-1679246480-aws-endpoint
linear-learner-2023-03-19-17-20-04-300-1679246405-aws-endpoint
ntm-2023-03-19-17-16-34-702-1679246195-aws-endpoint
sm-clarify-DEMO-clarify-model-19-03-2023-17-10--1679246123-1b57-1679246124-aws-endpoint
DEMO-xgb-churn-model-monitor-2023-03-19-1712-1679245978-aws-endpoint
sagemaker-scikit-learn-2023-03-19-17-11-24-292-1679245884-aws-endpoint
jumpstart-example-lightgbm-classificati-2023-03-19-17-09-07-745-1679245758-aws-endpoint
jumpstart-example-tensorflow-od1-ssd-re-2023-03-19-17-07-46-871-1679245734-aws-endpoint
linear-learner-2023-03-

blazingtext-2023-03-11-12-21-55-961-1678537317-aws-endpoint
rl-auto-scaling-2023-03-10-16-51-19-206-1678467081-aws-endpoint
sm-clarify-pipelines-ou6toabhsuby-AbaloneCreate-1678466658-ecb0-1678466658-aws-endpoint
sm-clarify-pipelines-ou6toabhsuby-AbaloneCreate-1678466655-8f61-1678466655-aws-endpoint
ss-notebook-demo-2023-03-10-16-41-20-432-1678466482-aws-endpoint
sm-clarify-DEMO-clarify-ll-model-10-03-2023-16--1678466479-a477-1678466480-aws-endpoint
sm-clarify-DEMO-clarify-ll-model-10-03-2023-16--1678465973-5762-1678465974-aws-endpoint
DEMO-xgb-churn-model-quality-monitor-2023-03-10-1630-1678465811-aws-endpoint
jumpstart-example-classic-gecko-ppp-endpoint-1678465771-aws-endpoint
DEMO-XGBoostEndpoint-2023-03-10-16-28-03-1678465683-aws-endpoint
jumpstart-example-autogluon-classificat-2023-03-10-16-22-20-497-1678465658-aws-endpoint
music-rec-endpoint-20230310-162556-1678465557-aws-endpoint
sagemaker-soln-documents--huggingface-t-2023-03-10-16-24-35-745-1678465477-aws-endpoint
blazingtext-2

womens-ecommerce-reviews-endpoint-19-02-2023-21-29-50-1676842218-aws-endpoint
KMeansEndpoint-2023-02-19-21-24-45-1676841886-aws-endpoint
jumpstart-example-churn-lgb-g-2023-02-19-21-23-17-499-1676841802-aws-endpoint
DEMO-XGBoostEndpoint-2023-02-19-21-20-51-1676841651-aws-endpoint
jumpstart-example-huggingface-tc-bert-b-2023-02-19-21-16-02-008-1676841397-aws-endpoint
sagemaker-inference-mxnet-2023-02-19-11-33-52-729-1676806432-aws-endpoint
sm-clarify-DEMO-clarify-model-19-02-2023-10-28--1676803726-eb07-1676803726-aws-endpoint
sm-clarify-DEMO-clarify-model-19-02-2023-10-28--1676802876-d936-1676802877-aws-endpoint
autogluon-inference-2023-02-19-10-31-02-709-1676802663-aws-endpoint
sagemaker-scikit-learn-2023-02-19-10-25-13-619-1676802314-aws-endpoint
rl-cart-pole-2023-02-19-10-25-10-262-1676802311-aws-endpoint
DEMO-linear-endpoint-202302191021-1676802088-aws-endpoint
ratings-features-music-rec-1676801613
ratings-features-music-rec-1676801613-feature-group-pipeline
ratings-features-music-re

jumpstart-example-infer-pytorch-od-nvid-2023-02-13-23-57-53-761-1676332813-aws-endpoint
sagemaker-scikit-learn-2023-02-14-00-00-06-220-1676332806-aws-endpoint
jumpstart-example-infer-mxnet-is-mask-r-2023-02-13-23-57-59-109-1676332764-aws-endpoint
mnist-feature-group-13-23-58-28-1676332709
mnist-feature-group-13-23-58-28-1676332709-feature-group-pipeline-version
mnist-feature-group-13-23-58-28-1676332709-feature-group-pipeline
linear-learner-2023-02-13-23-57-50-632-1676332671-aws-endpoint
mxnet-training-ml-c5-2023-02-13-23-57-43-728-1676332664-aws-endpoint
mnist-multi-container-ep-1676332644-aws-endpoint
linear-learner-2023-02-13-23-56-06-152-1676332566-aws-endpoint
sagemaker-soln-documents--a259eb-question-answering-endpoint-1676332341-aws-endpoint
DEMO-ic-lst-2022-12-28-22-58-27-002-43e55a08-1672270097-aws-endpoint
sm-epc-7342c923-c5e1-454c-bf07-5bba3e3440e7-1672267856-aws-endpoint
jumpstart-example-infer-model-txt2img-s-2022-12-28-22-27-44-741-1672267705-aws-endpoint
sagemaker-infere

## Use Case 3: Create an Action

Actions represent computational steps such as model builds, transformations, or training jobs.

In [6]:
model_build_action = Action.create(
    action_name=f"model-build-step-{unique_id}",
    action_type="ModelBuild",
    source_uri=unique_id,
    properties={"Example": "Metadata"},
)

print(f"Created action: {model_build_action.action_name}")

Created action: model-build-step-1784619225


## Use Case 4: Create Associations

Associations are directed edges in the lineage graph. The `association_type` can be `Produced`, `DerivedFrom`, `AssociatedWith`, or `ContributedTo`.

In [7]:
context_action_association = Association.create(
    source_arn=ml_workflow_context.context_arn,
    destination_arn=model_build_action.action_arn,
    association_type="AssociatedWith",
)

print("Association created between context and action")

Association created between context and action


## Use Case 5: Traverse Associations

Query incoming and outgoing associations to understand entity relationships.

In [8]:
# List incoming associations to the action
incoming_associations = Association.list(destination_arn=model_build_action.action_arn)
for association in incoming_associations:
    print(
        f"{model_build_action.action_name} has an incoming association from {association.source_name}"
    )

# List outgoing associations from the context
outgoing_associations = Association.list(source_arn=ml_workflow_context.context_arn)
for association in outgoing_associations:
    print(
        f"{ml_workflow_context.context_name} has an outgoing association to {association.destination_name}"
    )

model-build-step-1784619225 has an incoming association from machine-learning-workflow-1784619225


machine-learning-workflow-1784619225 has an outgoing association to model-build-step-1784619225


## Use Case 6: Create Artifacts

Artifacts represent URI-addressable objects or data such as datasets, labels, or trained models.

In [9]:
# Create input data artifacts
input_test_images = Artifact.create(
    artifact_name="mnist-test-images",
    artifact_type="TestData",
    source_types=[{"SourceIdType": "Custom", "Value": unique_id}],
    source_uri=f"https://sagemaker-example-files-prod-{region}.s3.amazonaws.com/datasets/image/MNIST/t10k-images-idx3-ubyte.gz",
)

input_test_labels = Artifact.create(
    artifact_name="mnist-test-labels",
    artifact_type="TestLabels",
    source_types=[{"SourceIdType": "Custom", "Value": unique_id}],
    source_uri=f"https://sagemaker-example-files-prod-{region}.s3.amazonaws.com/datasets/image/MNIST/t10k-labels-idx1-ubyte.gz",
)

print(f"Created artifact: {input_test_images.artifact_name}")
print(f"Created artifact: {input_test_labels.artifact_name}")

Created artifact: mnist-test-images
Created artifact: mnist-test-labels


In [10]:
# Create output model artifact
output_model = Artifact.create(
    artifact_name="mnist-model",
    artifact_type="Model",
    source_types=[{"SourceIdType": "Custom", "Value": unique_id}],
    source_uri=f"https://sagemaker-example-files-prod-{region}.s3.amazonaws.com/datasets/image/MNIST/model/tensorflow-training-2020-11-20-23-57-13-077/model.tar.gz",
)

print(f"Created artifact: {output_model.artifact_name}")

Created artifact: mnist-model


## Use Case 7: Link Artifacts to Actions

Associate data artifacts as inputs to the action, and the action output to the model artifact.

In [11]:
# Associate input data with the model build action
Association.create(
    source_arn=input_test_images.artifact_arn,
    destination_arn=model_build_action.action_arn,
)
Association.create(
    source_arn=input_test_labels.artifact_arn,
    destination_arn=model_build_action.action_arn,
)

# Associate the action with the output model
Association.create(
    source_arn=model_build_action.action_arn,
    destination_arn=output_model.artifact_arn,
)

print("Lineage graph complete: inputs -> action -> output")

Lineage graph complete: inputs -> action -> output


## Cleanup

Delete all lineage entities created in this notebook. Associations must be removed before their source or destination entities can be deleted.

In [12]:
def delete_associations(arn):
    """Delete all incoming and outgoing associations for an entity."""
    for summary in Association.list(destination_arn=arn):
        assct = Association(
            source_arn=summary.source_arn,
            destination_arn=summary.destination_arn,
            sagemaker_session=sagemaker_session,
        )
        assct.delete()

    for summary in Association.list(source_arn=arn):
        assct = Association(
            source_arn=summary.source_arn,
            destination_arn=summary.destination_arn,
            sagemaker_session=sagemaker_session,
        )
        assct.delete()


def delete_lineage_data():
    """Delete all lineage entities created in this notebook."""
    print(f"Deleting context {ml_workflow_context.context_name}")
    delete_associations(ml_workflow_context.context_arn)
    ctx = Context(
        context_name=ml_workflow_context.context_name,
        sagemaker_session=sagemaker_session,
    )
    ctx.delete()

    print(f"Deleting action {model_build_action.action_name}")
    delete_associations(model_build_action.action_arn)
    actn = Action(
        action_name=model_build_action.action_name,
        sagemaker_session=sagemaker_session,
    )
    actn.delete()

    for artifact in [input_test_images, input_test_labels, output_model]:
        print(f"Deleting artifact {artifact.artifact_arn} {artifact.artifact_name}")
        delete_associations(artifact.artifact_arn)
        artfct = Artifact(
            artifact_arn=artifact.artifact_arn,
            sagemaker_session=sagemaker_session,
        )
        artfct.delete()


delete_lineage_data()
print("Cleanup complete")

Deleting context machine-learning-workflow-1784619225


Deleting action model-build-step-1784619225


Deleting artifact arn:aws:sagemaker:us-west-2:674622101542:artifact/caed1194e61569e7d6d8610f74aa5a1a mnist-test-images


Deleting artifact arn:aws:sagemaker:us-west-2:674622101542:artifact/33fbf183e6f76b348244518d3e37b815 mnist-test-labels


Deleting artifact arn:aws:sagemaker:us-west-2:674622101542:artifact/0b150c6c83a42c1c5474b5e17faf1143 mnist-model


Cleanup complete
